# 00 — Ambiente de reserva no Google Colab (Frente A)

Use só se a RTX 5060 não estiver disponível. Ambiente → *Alterar o tipo de ambiente de execução* → **GPU T4**.

Credenciais do Kaggle: **não cole o token no notebook**. Cadastre `KAGGLE_USERNAME` e `KAGGLE_KEY`
no painel **Secrets** (ícone de chave, à esquerda) e libere o acesso para este notebook.
Antes, aceite as regras da competição FER-2013 no site do Kaggle.

In [ ]:
!nvidia-smi

## 1. Google Drive para checkpoints e cache de pesos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/seminario-emocoes'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/torch_home', exist_ok=True)
os.environ['TORCH_HOME'] = f'{DRIVE_ROOT}/torch_home'   # pesos do ImageNet ficam no Drive

## 2. Código e dependências

In [ ]:
%cd /content
!git clone https://github.com/KauaSandes/seminario-reconhecimento-emocoes.git
%cd seminario-reconhecimento-emocoes
# O Colab já traz torch/torchvision com CUDA; instale só o resto.
!pip install -q -r requirements.txt
# Checkpoints vão direto para o Drive
!ln -sfn {DRIVE_ROOT}/checkpoints checkpoints

## 3. Dados (mesmo script da Frente B usado na máquina local)

In [ ]:
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
!python scripts/download_data.py
!sha256sum data/raw/fer2013/fer2013.csv   # comparar com o hash publicado no README

## 4. Verificação do ambiente e cache dos pesos

In [ ]:
!python scripts/check_env.py
!python scripts/cache_pretrained.py

## 5. Teste de fumaça do pipeline

In [ ]:
!python scripts/smoke_test.py --mode small && python scripts/smoke_test.py --mode resnet